# Demo 2 pipeline bằng endpoint miễn phí

## 1. Setup

In [ ]:
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings, logging
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from pipelines.model_clients import GeminiClient, ModelClient, _usage_dict, _image_to_data_uri
from pipelines.ocr_engines import EasyOCREngine
from pipelines.vlm_direct import VLMDirectPipeline
from pipelines.ocr_llm import OCRLLMPipeline
from pipelines.prompt_config import load_field_config, load_prompt_template, field_names
from evaluation.metrics import exact_match, token_f1

assert os.getenv("GOOGLE_API_KEY"), "Chưa có GOOGLE_API_KEY trong .env"
print("ROOT =", ROOT)

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
names = [m.name.removeprefix("models/") for m in genai.list_models()
         if "generateContent" in m.supported_generation_methods]
print([n for n in names if n.startswith(("gemini-2.5", "gemma"))])

## 2. Chọn mẫu

In [ ]:
DATASET = "EN/Typed-Invoice"
DOC_NAME = None          # None = lấy mẫu đầu tiên của tập

VLM_MODEL = "gemini-2.5-flash"
TEXT_MODEL = "gemma-4-26b-a4b-it"
MIN_INTERVAL = 5.0       # giãn cách giữa 2 lần gọi để không vượt giới hạn free tier

ds_dir = ROOT / "data-processed" / DATASET
gt_files = sorted((ds_dir / "ground_truth").glob("*.json"))
gt_path = next(p for p in gt_files if DOC_NAME is None or p.stem == DOC_NAME)
gt = json.loads(gt_path.read_text(encoding="utf-8"))
image_path = ds_dir / "img" / Path(gt["image_path"]).name

field_config = load_field_config(ROOT / "config" / "fields" / f"{gt['doc_type']}.json")
fields = field_names(field_config)
vlm_prompt = load_prompt_template(ROOT / "config" / "prompts" / "vlm_direct.md")
ocr_prompt = load_prompt_template(ROOT / "config" / "prompts" / "ocr_llm.md")
ocr_lang = ["vi", "en"] if gt["lang"] == "VN" else "en"

print("Mẫu:", gt["doc_name"], "| loại:", gt["doc_type"], "| ngôn ngữ:", gt["lang"])
print("Trường:", fields)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

with Image.open(image_path) as img:
    plt.figure(figsize=(6, 8)); plt.imshow(img); plt.axis("off"); plt.show()

In [ ]:
import pandas as pd

runs = []

def run(label, pipeline):
    t0 = time.perf_counter()
    res = pipeline.extract(image_path)
    latency = time.perf_counter() - t0
    em = exact_match(res.fields, gt["fields"], fields)
    f1 = token_f1(res.fields, gt["fields"], fields)
    runs.append({"label": label, "fields": res.fields, "em": em, "f1": f1,
                 "latency": latency, "usage": res.usage})
    df = pd.DataFrame({"ground_truth": gt["fields"], "predicted": res.fields,
                       "exact_match": em, "token_f1": f1})
    print(f"{label} | độ trễ {latency:.1f}s | tokens {res.usage}")
    print(f"EM = {sum(em.values())/len(em):.0%} | F1 = {sum(f1.values())/len(f1):.3f}")
    return df

## 3. Pipeline A: VLM Direct

In [ ]:
vlm_client = GeminiClient(model=VLM_MODEL, api_key=os.environ["GOOGLE_API_KEY"],
                          min_interval_sec=MIN_INTERVAL)
pipe_a = VLMDirectPipeline(vlm_client, vlm_prompt, field_config)
run(f"A | {VLM_MODEL}", pipe_a)

## 4. Pipeline B: OCR-LLM

In [ ]:
ocr = EasyOCREngine(languages=ocr_lang, gpu=False)
ocr_text = ocr.extract_text(image_path)
print(f"OCR ra {len(ocr_text)} ký tự. Trích 500 ký tự đầu:\n")
print(ocr_text[:500])

In [ ]:
from pipelines.model_clients import _gemini_extract_usage

# Gemma 4 trả đoạn suy luận rồi mới đến JSON, chỉ lấy phần cuối
class GemmaClient(GeminiClient):
    def generate(self, prompt, image=None):
        contents = [image, prompt] if image is not None else prompt
        response = self._client.generate_content(contents)
        parts = response.candidates[0].content.parts if response.candidates else []
        text = parts[-1].text if parts else None
        return (text or None), _gemini_extract_usage(response)

text_client = GemmaClient(model=TEXT_MODEL, api_key=os.environ["GOOGLE_API_KEY"],
                           min_interval_sec=MIN_INTERVAL)
pipe_b = OCRLLMPipeline(text_client, ocr_prompt, field_config, ocr_engine=ocr)
run(f"B | EasyOCR + {TEXT_MODEL}", pipe_b)

## 5. Endpoint tương thích OpenAI (tuỳ chọn)

In [ ]:
from openai import OpenAI

class OpenAICompatibleClient(ModelClient):
    provider = "openai_compatible"

    def __init__(self, model, base_url, api_key, supports_vision=False, min_interval_sec=0.0):
        self.name = model
        self.supports_vision = supports_vision
        self.min_interval_sec = min_interval_sec
        self._client = OpenAI(base_url=base_url, api_key=api_key)

    def generate(self, prompt, image=None):
        content = prompt
        if image is not None:
            content = [{"type": "text", "text": prompt},
                       {"type": "image_url", "image_url": {"url": _image_to_data_uri(image)}}]
        r = self._client.chat.completions.create(
            model=self.name, messages=[{"role": "user", "content": content}], temperature=0)
        u = r.usage
        usage = _usage_dict(u.prompt_tokens, u.completion_tokens, None, u.total_tokens) if u else None
        return (r.choices[0].message.content or None), usage

In [ ]:
# Groq / OpenRouter (https://openrouter.ai/api/v1) / Ollama local (http://localhost:11434/v1)
BASE_URL = "https://api.groq.com/openai/v1"
API_KEY = os.getenv("GROQ_API_KEY")
MODEL = "llama-3.3-70b-versatile"   # đổi theo danh sách model của nhà cung cấp

if API_KEY:
    extra = OpenAICompatibleClient(MODEL, BASE_URL, API_KEY, min_interval_sec=MIN_INTERVAL)
    display(run(f"B | EasyOCR + {MODEL}", OCRLLMPipeline(extra, ocr_prompt, field_config, ocr_engine=ocr)))
else:
    print("Bỏ qua: chưa có key cho endpoint này trong .env")

## 6. So sánh với benchmark 05/2026

In [ ]:
rows = []
for p in sorted((ROOT / "results").glob(f"*/*/{gt['doc_name']}.json")):
    r = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"nguồn": "benchmark 05/2026", "pipeline": r["pipeline"], "model": r["model"],
                 "EM": r["overall_exact_match"], "F1": r["overall_token_f1"],
                 "latency_s": r["latency_sec"], "tokens": (r.get("usage") or {}).get("total_tokens")})
for x in runs:
    rows.append({"nguồn": "chạy lại hôm nay", "pipeline": x["label"].split(" | ")[0], "model": x["label"].split(" | ")[1],
                 "EM": sum(x["em"].values()) / len(x["em"]), "F1": sum(x["f1"].values()) / len(x["f1"]),
                 "latency_s": round(x["latency"], 2), "tokens": (x["usage"] or {}).get("total_tokens")})
pd.DataFrame(rows).sort_values(["nguồn", "pipeline", "F1"], ascending=[True, True, False]).reset_index(drop=True)

In [ ]:
pd.DataFrame({"ground_truth": gt["fields"], **{x["label"]: x["fields"] for x in runs}})